# **Cloud-Native GRIB2: High-Performance Access with grib2io and VirtualiZarr**

This notebook demonstrates how to efficiently access GRIB2 data stored on AWS S3 without streaming the entire file. We'll leverage standard `.idx` sidecar files to build a virtual Xarray dataset instantly.

## **The GRIB2 Challenge**

GRIB2 is a streaming format without a central metadata header. To create a virtual Zarr manifest, a standard scanner must read the file from start to finish to find message boundaries. For a multi-gigabyte file on S3, this normally means streaming all that data just to find the offsets.

### **The Solution: Standard sidecar Indices**
Public datasets (like NOAA GFS) often provide text-based `.idx` files alongside the GRIB files. `grib2io` can now parse these `.idx` files to extract byte offsets directly. This allows it to jump to each message and read only the small metadata headers (Sections 0-5), completely bypassing the large data payloads (Section 7) during the scanning phase.

In [ ]:
import xarray as xr
import grib2io
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser
import s3fs
import json
import os

## **1. Locating GFS Data on S3**

We'll use a public GFS dataset from the NOAA Open Data Program on AWS. Note the presence of the `.idx` file.

In [ ]:
# Example S3 path for a GFS file
s3_bucket = "noaa-gfs-bdp-pds"
s3_path = "gfs.20240501/00/atmos/gfs.t00z.pgrb2.0p25.f000"
s3_url = f"s3://{s3_bucket}/{s3_path}"
# Standard sidecar index: s3://noaa-gfs-bdp-pds/gfs.20240501/00/atmos/gfs.t00z.pgrb2.0p25.f000.idx

## **2. Building the Virtual Manifest via Index**

By leveraging the `.idx` file, `grib2io` maps out the chunks instantly. The `ReferenceGenerator` uses this to create a Kerchunk manifest.

In [ ]:
from grib2io.kerchunk import ReferenceGenerator

# ReferenceGenerator handles the remote URL and automatically
# discovers the .idx sidecar to accelerate the scan.
gen = ReferenceGenerator(s3_url)

# This step is now extremely fast as it avoids streaming data payloads.
manifest = gen.generate()

manifest_path = "gfs_s3_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f)

## **3. Ingesting into VirtualiZarr**

Load the S3-backed manifest into VirtualiZarr for metadata manipulation.

In [ ]:
vds = open_virtual_dataset(manifest_path, parser=KerchunkJSONParser(), loadable_variables=[])

display(vds)

## **4. Targeted Data Access**

Accessing the virtual dataset triggers precise byte-range requests. For example, slicing a small region only downloads the bytes for the specific GRIB message needed.

In [ ]:
if "TMP" in vds.data_vars:
    # This only downloads the few hundred kilobytes for the surface temperature message.
    temp_slice = vds.TMP.isel(y=slice(100, 200), x=slice(100, 200))
    display(temp_slice)